In [ ]:
from datetime import datetime, timedelta

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression

In [ ]:
# Constants
RISK_FREE_RATE = 0.04
SEED = 0 # for reproducibility

END_DATE = datetime.today()
START_DATE = END_DATE - timedelta(days=365*2)

In [ ]:
np.random.seed(SEED)

tickers = ["MU", "EQIX", "ETN", "AMD", "EL", "WDC", "ANET"]

In [ ]:
class efficientFrontier:
    def __init__(self,
                 tickers,
                 start_date,
                 end_date
                 ):
        self.start_date = start_date
        self.end_date = end_date

        self.tickers = tickers
        # download data from yfinance
        prices = yf.download(self.tickers, start=self.start_date, end=self.end_date, interval="1d")["Close"]

        self.returns = prices.pct_change().dropna()
        self.mean_returns = self.returns.mean()
        self.covariance = self.returns.cov()
        self.correlation = self.returns.corr()

        self.mean_returns_annualized = self.mean_returns * 252
        self.sigma = self.covariance * 252

    def monte_carlo(self, n_portfolios: int = 200000):
        results = []

        for _ in range(n_portfolios):
            weights = np.random.dirichlet(len(self.tickers))

            portfolio_return = np.sum(self.mean_returns_annualized * weights)
            portfolio_volatility = np.sqrt(
                np.dot(weights.T, np.dot(self.sigma, weights))
            )
            sharpe_ratio = (portfolio_return - RISK_FREE_RATE) / portfolio_volatility \
                if portfolio_volatility != 0 else 0
            
            results.append([portfolio_return, portfolio_volatility, sharpe_ratio] + list(weights))

        cols = ["Return", "Volatility", "Sharpe Ratio"] + self.tickers
        self.portfolios = pd.DataFrame(results, columns=cols)

        self.max_sharpe = self.portfolios.loc[self.portfolios["Sharpe Ratio"].idxmax()]
        self.min_volatility = self.portfolios.loc[self.portfolios["Volatility"].idxmin()]
    
    def regression(self):
        raise NotImplementedError
    
    def plot_monte_carlo(self):
        raise NotImplementedError
    
    def plot_regression(self):
        raise NotImplementedError
    
    def plot_correlation_matrix(self):
        raise NotImplementedError